# Notebook 3: Nonlinear Elasticity, Murnaghan Theory & the Acoustoelastic Effect## Theory connecting strain to dv/v through third-order elastic constants### Key References- Murnaghan, F. D. (1937). Finite deformations of an elastic solid. *Am. J. Math.*, 59(2), 235–260.- Clements, T. & Denolle, M. A. (2023). JGR Solid Earth, 128, e2022JB025553.- Tromp, J. & Trampert, J. (2018). Effects of induced stress on seismic forward modelling and inversion. *GJI*, 213, 851–867.- Richter, T. et al. (2014). JGR Solid Earth, 119, 4747–4765.- Sens-Schönfelder, C. & Eulenfeld, T. (2019). *PRL*, 122, 138501.

In [ ]:
import numpy as npimport matplotlib.pyplot as pltplt.rcParams.update({    'font.size': 12, 'axes.labelsize': 14, 'figure.dpi': 150,    'font.family': 'serif', 'mathtext.fontset': 'cm',    'axes.grid': True, 'grid.alpha': 0.3})print("Environment ready.")

## 1. Murnaghan's Third-Order Elastic EnergyThe elastic potential energy density expanded to third order (Murnaghan, 1937):$$\rho_0 \phi = \frac{\lambda + 2\mu}{2} I_1^2 - 2\mu I_2 + l I_1^3 + m I_1 I_2 + n I_3$$where $I_1, I_2, I_3$ are the strain invariants, and $l, m, n$ are the **Murnaghan third-order elastic constants** (TOE constants).The stress tensor to second-order accuracy becomes:$$T_s^r = \{\lambda I_1 + (3l + m - \lambda)I_1^2 + mI_2\}\delta_s^r + \{2\mu - (m + 2\lambda + 2\mu)I_1\}\varepsilon_s^r - 4\mu\varepsilon_a^r\varepsilon_s^a + nI_3\zeta_s^r$$### The Acoustoelastic EffectUnder nonlinear elasticity (Clements & Denolle, 2023, Eq. 1–4):$$\sigma = M(\epsilon + \beta\epsilon^2 + \ldots)$$The **acoustoelastic parameter** $\beta$ captures the sensitivity of velocity to strain:$$\beta = \frac{3}{2} + \frac{l + 2m}{\lambda + 2\mu}$$The velocity change due to volumetric strain $\epsilon_{kk}$ is:$$\frac{dv}{v} = \beta \, \epsilon_{kk}$$

In [ ]:
# === Acoustoelastic Parameter and dv/v vs Strain ===
def acoustoelastic_beta(l, m, lam, mu):
    """
    Compute acoustoelastic parameter beta.
    Clements & Denolle (2023), Eq. 2.
    """
    return 1.5 + (l + 2*m) / (lam + 2*mu)

# Typical values from literature
materials = {
    'Steel':         {'beta': -10,   'color': 'gray'},
    'Concrete':      {'beta': -50,   'color': 'brown'},
    'Barre Granite': {'beta': -500,  'color': 'blue'},
    'Marble':        {'beta': -1000, 'color': 'purple'},
    'Fontainebleau Sst': {'beta': -10000, 'color': 'orange'},
}

# dv/v as function of volumetric strain
epsilon_kk = np.linspace(-1e-5, 1e-5, 500)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel (a): dv/v vs strain for different materials
#
# NOTE: dv/v = beta * eps_kk is the LINEAR acoustoelastic law and is only valid
# while |dv/v| stays small (|dv/v| < ~0.5%). For large |beta| (e.g. Fontainebleau
# sandstone, beta ~ -1e4) the line reaches several % at eps ~ 1e-5, which is an
# unphysical extrapolation -- higher-order terms dominate there. We therefore
# (i) clip the y-axis to the linear-theory range, and (ii) shade the validity band.
ax = axes[0]
dvv_valid = 0.5  # % -- approximate upper bound of linear acoustoelastic validity
for name, props in materials.items():
    beta = props['beta']
    dvv = beta * epsilon_kk * 100  # percent
    ax.plot(epsilon_kk * 1e6, dvv, lw=2, color=props['color'], label=f'{name} ($\\beta$ = {beta})')
ax.axhspan(-dvv_valid, dvv_valid, color='green', alpha=0.10,
           label=f'Linear theory valid (|dv/v| $\\lesssim$ {dvv_valid}%)')
ax.set_xlabel('Volumetric strain $\\epsilon_{{kk}}$ [microstrain]')
ax.set_ylabel('dv/v [%]')
ax.set_ylim(-dvv_valid*1.4, dvv_valid*1.4)   # do NOT display the absurd -10% extrapolation
ax.set_title('(a) dv/v vs volumetric strain\n(linear acoustoelastic law; valid for |dv/v| $\\lesssim$ 0.5%)')
ax.legend(fontsize=9)
ax.axhline(0, color='k', ls='-', alpha=0.3)
ax.axvline(0, color='k', ls='-', alpha=0.3)

# Panel (b): Phase diagram of beta
ax = axes[1]
lam_mu_ratios = np.linspace(0.5, 3.0, 50)
mu_fixed = 30e9
for l_val in [-100e9, -500e9, -1000e9]:
    for m_val in [-200e9, -800e9]:
        betas = []
        for ratio in lam_mu_ratios:
            lam = ratio * mu_fixed
            b = acoustoelastic_beta(l_val, m_val, lam, mu_fixed)
            betas.append(b)
        ax.plot(lam_mu_ratios, betas, lw=1, alpha=0.5)
# Highlight typical range
ax.axhspan(-10000, -100, alpha=0.1, color='blue', label='Typical rock range')
ax.set_xlabel('$\\lambda/\\mu$ ratio')
ax.set_ylabel('$\\beta$ (acoustoelastic parameter)')
ax.set_title('(b) $\\beta$ sensitivity to Lamé ratio & TOE constants')
ax.set_yscale('symlog', linthresh=10)
ax.legend()

fig.suptitle('Nonlinear Elasticity: Acoustoelastic Effect\n'
             '(Murnaghan, 1937; Clements & Denolle, 2023)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('fig08_acoustoelastic.png', bbox_inches='tight')
plt.show()

print("\nReported β values (Clements & Denolle, 2023):")
for name, props in materials.items():
    print(f"  {name:25s}: β ≈ {props['beta']}")
print("\nNote: the linear law dv/v = β·ε_kk is only valid for |dv/v| ≲ 0.5%;")
print("beyond that, higher-order (Murnaghan) terms must be retained.")

## 2. Detecting Nonlinear Elasticity from dv/v and StrainA key diagnostic for nonlinear elasticity is the **linear relationship** betweendv/v and volumetric strain $\epsilon_{kk}$:$$\frac{dv}{v} = \beta \, \epsilon_{kk}$$If the relationship deviates from linear, this may indicate:1. **Higher-order nonlinearity** (beyond third-order)2. **Hysteretic/mesoscopic nonlinearity** (slow dynamics)3. **Crack opening/closing** thresholdsSens-Schönfelder & Eulenfeld (2019) demonstrated detection of in-situ nonlinearelasticity using Earth tides — the tidal strain provides a known, periodic forcing.

In [ ]:
# === dv/v vs Strain: Linear vs Nonlinear ===
# Simulated strain from tidal + tectonic + environmental
np.random.seed(42)
t_hours = np.arange(0, 365*24, 1)  # 1 year, hourly
t_days = t_hours / 24
# Tidal strain (M2 + O1 approximation)
omega_M2 = 2 * np.pi / 12.42  # hours
omega_O1 = 2 * np.pi / 25.82
epsilon_tidal = 50e-9 * np.cos(omega_M2 * t_hours) + 30e-9 * np.cos(omega_O1 * t_hours)
# Thermoelastic strain (annual)
epsilon_thermal = 2e-6 * np.cos(2*np.pi*t_days/365.25)
# Total strain
epsilon_total = epsilon_tidal + epsilon_thermal
# dv/v with different rheologies
beta_linear = -500
# Quadratic acoustoelastic coefficient. Chosen so the quadratic term is a
# visible (~20%) fraction of the linear term at the max strain shown
# (eps_max ~ 2e-6): gamma*eps^2 ~ 0.2*|beta|*eps  =>  gamma ~ 0.2*|beta|/eps_max.
# (A textbook gamma~1e5 would be ~1e3x too small to see here.)
beta_quadratic = 5e7  # higher-order term
dvv_linear = beta_linear * epsilon_total
dvv_nonlinear = beta_linear * epsilon_total + beta_quadratic * epsilon_total**2
dvv_hysteretic = beta_linear * epsilon_total * (1 + 0.3 * np.sign(np.gradient(epsilon_total)))

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Time series
ax = axes[0, 0]
ax.plot(t_days[:720], epsilon_total[:720]*1e9, 'k-', lw=0.5)
ax.set_xlabel('Time [days]')
ax.set_ylabel('Strain [nanostrain]')
ax.set_title('(a) Total strain (tidal + thermal)')

# dv/v time series
ax = axes[0, 1]
ax.plot(t_days[:720], dvv_linear[:720]*100, 'b-', lw=0.5, alpha=0.7, label='Linear')
ax.plot(t_days[:720], dvv_nonlinear[:720]*100, 'r-', lw=0.5, alpha=0.7, label='Nonlinear')
ax.set_xlabel('Time [days]')
ax.set_ylabel('dv/v [%]')
ax.set_title('(b) dv/v: linear vs nonlinear response')
ax.legend()

# Crossplot: dv/v vs strain (LINEAR)
ax = axes[1, 0]
subsample = slice(0, 720*24, 10)
ax.scatter(epsilon_total[subsample]*1e9, dvv_linear[subsample]*1e4, 
           c=t_days[subsample], cmap='viridis', s=1, alpha=0.5)
ax.set_xlabel('Strain [nanostrain]')
ax.set_ylabel('dv/v [10⁻⁴]')
ax.set_title('(c) Linear: dv/v vs strain')
# Fit line
coeffs = np.polyfit(epsilon_total[subsample], dvv_linear[subsample], 1)
eps_fit = np.linspace(epsilon_total.min(), epsilon_total.max(), 100)
ax.plot(eps_fit*1e9, np.polyval(coeffs, eps_fit)*1e4, 'r-', lw=2, 
        label=f'β = {coeffs[0]:.0f}')
ax.legend()

# Crossplot: dv/v vs strain (NONLINEAR)
ax = axes[1, 1]
ax.scatter(epsilon_total[subsample]*1e9, dvv_nonlinear[subsample]*1e4,
           c=t_days[subsample], cmap='viridis', s=1, alpha=0.5)
ax.set_xlabel('Strain [nanostrain]')
ax.set_ylabel('dv/v [10⁻⁴]')
ax.set_title('(d) Nonlinear: dv/v vs strain (curvature visible)')
coeffs_nl = np.polyfit(epsilon_total[subsample], dvv_nonlinear[subsample], 2)
ax.plot(eps_fit*1e9, np.polyval(coeffs_nl, eps_fit)*1e4, 'r-', lw=2,
        label=f'Quadratic fit')
ax.legend()

fig.suptitle('Diagnosing Nonlinear Elasticity from dv/v–Strain Crossplots\n'
             '(cf. Sens-Schönfelder & Eulenfeld, 2019)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('fig09_nonlinear_detection.png', bbox_inches='tight')
plt.show()

## 3. Murnaghan Pressure-Volume RelationFrom Murnaghan (1937), the pressure-volume relation under hydrostatic compression:$$p = af + bf^2, \quad f = \frac{1}{2}\left[(V_0/V)^{2/3} - 1\right]$$where $a = 3\lambda + 2\mu$ and $b = 15\lambda + 10\mu - 27l - 9m - n$.With only two constants ($\lambda, \mu$): $b = 5a$, giving a one-parameter formula.This predicts velocity increases with pressure (depth) — the fundamental observationunderlying the stress-sensitivity of seismic velocities.

In [ ]:
# === Murnaghan Equation of State ===def murnaghan_pressure(V_V0, lam, mu, l=0, m=0, n=0):    """    Murnaghan (1937) pressure-volume relation.    p = a*f + b*f^2  where f = 0.5*((V0/V)^(2/3) - 1)    """    f = 0.5 * ((1.0/V_V0)**(2.0/3.0) - 1.0)    a = 3*lam + 2*mu    b = 15*lam + 10*mu - 27*l - 9*m - n    return a*f + b*f**2# Compare two-constant vs five-constant MurnaghanV_V0 = np.linspace(0.85, 1.0, 200)lam, mu = 40e9, 30e9  # typical crustal valuesfig, axes = plt.subplots(1, 2, figsize=(14, 6))# Panel (a): Two-constant theoryax = axes[0]p_2const = murnaghan_pressure(V_V0, lam, mu) / 1e9ax.plot(V_V0, p_2const, 'b-', lw=2.5, label='2-constant (l=m=n=0)')# Five-constant with typical TOE valuesfor l_val, m_val, n_val, label in [    (-1000e9, -500e9, -200e9, 'Large TOE'),    (-200e9, -100e9, -50e9,   'Moderate TOE'),    (-50e9, -30e9, -10e9,     'Small TOE'),]:    p_5const = murnaghan_pressure(V_V0, lam, mu, l_val, m_val, n_val) / 1e9    ax.plot(V_V0, p_5const, '--', lw=2, label=label)ax.set_xlabel('V/V₀')ax.set_ylabel('Pressure [GPa]')ax.set_title('(a) Murnaghan equation of state')ax.legend()# Panel (b): Velocity vs pressure (from Murnaghan)ax = axes[1]pressures_MPa = np.linspace(0, 100, 200)# V_p = sqrt((lam + 2*mu + pressure_correction) / rho)rho = 2700  # kg/m^3for drv in [5, 50, 200, 500]:    # Using d(rho*v^2)/d(sigma_c) = drv    Vp = np.sqrt((lam + 2*mu + drv * pressures_MPa*1e6) / rho)    ax.plot(pressures_MPa, Vp/1e3, lw=2, label=f'$\\partial(\\rho v^2)/\\partial\\sigma_c$ = {drv}')ax.set_xlabel('Confining pressure [MPa]')ax.set_ylabel('P-wave velocity [km/s]')ax.set_title('(b) Velocity-pressure relation')ax.legend()fig.suptitle('Murnaghan (1937) Finite Deformation Theory\n'             'Foundation of stress-velocity sensitivity', fontsize=14, y=1.02)plt.tight_layout()plt.savefig('fig10_murnaghan_EOS.png', bbox_inches='tight')plt.show()